# The TLS904_NUTS table 

The `TLS904_NUTS` table is one of the reference tables in PATSTAT Global. It belongs to the 900-series tables, which mainly contain data originating from external sources outside the EPO. The purpose of this table is to provide standardised regional codes and related metadata based on the Nomenclature of Territorial Units for Statistics (NUTS). The NUTS system is a hierarchical geographic classification developed by the European Union (EU) to divide its territory into regions for statistical and analytical purposes. 
This reference table plays a key role in analysing the geographical distribution of patent-related activities, as it allows users to identify which specific regions are most frequently represented in patent data. Regional NUTS information, together with other attributes such as the sector of activity (`PSN_SECTOR`), is also available in the `TLS206_PERSON` table. The `TLS904_NUTS` table is directly linked to `TLS206_PERSON` via the `NUTS` attribute.

The NUTS regional data used in this table were provided by ECOOM (KU Leuven) and are based on publicly available Eurostat data.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS904_NUTS
from sqlalchemy import case, func, select, distinct, and_

# Initialise the PATSTAT client
patstat = PatstatClient(env="TEST")

# Access ORM
db = patstat.orm()

## Key attributes of the TLS904_NUTS table

### NUTS
This attribute contains the NUTS code as defined by Eurostat. In the table `TLS206_PERSON`, this field remains empty for countries that are not covered by NUTS codes. Since the PATSTAT 2020 Autumn Edition, specific person records have been enhanced with values from the OECD REGPAT Database (January 2020). These enriched records are distinguished by a `NUTS_LEVEL` value of **4**.
The data generally follows NUTS version 2013 (with minor additions). However, for the United Kingdom, the London area utilizes NUTS version 2010.

### NUTS_LABEL
This field contains the name of the NUTS region corresponding to the code. The name is provided in the original language of the region and follows the NUTS version 2013 standard.

### NUTS_LEVEL
This field indicates the hierarchical level of regionalisation associated with the code stored in the NUTS attribute. The level is represented by a single-digit number and reflects the most detailed geographic information available for a given record. Whenever possible, NUTS codes are assigned at the finest level of regional detail supported by the underlying address data. When detailed regional information is not available, a higher-level or default value is used instead. The specific meanings of the values are as follows:

| Value | Meaning |
| :--- | :--- |
| **0** | The NUTS code identifies an entire state (country level). |
| **1, 2, 3** | These represent the official NUTS levels (defining decreasing sizes of regions). |
| **4** | This level is equivalent to NUTS level 3, but specifically indicates that the code was provided by the OECD REGPAT Database. |
| **9** | Indicates that no NUTS code has been assigned. |

In [2]:
q = (
    db.query(
        TLS904_NUTS.nuts,
        TLS904_NUTS.nuts_label,
    )
    .filter(
        TLS904_NUTS.nuts_level == 0
    )
    .order_by(TLS904_NUTS.nuts)
)

res = patstat.df(q)
res

,nuts,nuts_label
0,AT,ÖSTERREICH
1,BE,BELGIQUE-BELGIË
2,BG,БЪЛГАРИЯ (BULGARIA)
3,CH,SCHWEIZ/SUISSE/SVIZZERA
4,CY,ΚΥΠΡΟΣ (KYPROS)
5,CZ,ČESKÁ REPUBLIKA
6,DE,DEUTSCHLAND
7,DK,DANMARK
8,EE,EESTI
9,EL,ΕΛΛΑΔΑ (ELLADA)


To examine the NUTS structure within a specific country, we can subset NUTS codes by their initial characters, which reflect the higher-level territorial classifications. By doing so, we can reconstruct the hierarchical organisation of NUTS regions. For example, in the following analysis we explore the NUTS codes for Italy, expanding them across selected NUTS levels to illustrate how regional, and sub-regional classifications are structured.

In [3]:
q = (
    db.query(
        TLS904_NUTS.nuts,
        TLS904_NUTS.nuts_label,
    )
    .filter(
        and_(
            func.substr(TLS904_NUTS.nuts, 1, 2) == "IT",
            TLS904_NUTS.nuts_level == 1
        )
    )
    .order_by(TLS904_NUTS.nuts)
)

res = patstat.df(q)
res

,nuts,nuts_label
0,ITC,NORD-OVEST
1,ITF,SUD
2,ITG,ISOLE
3,ITH,NORD-EST
4,ITI,CENTRO (IT)
5,ITZ,EXTRA-REGIO NUTS 1


In [4]:
q = (
    db.query(
        TLS904_NUTS.nuts,
        TLS904_NUTS.nuts_label,
    )
    .filter(
        and_(
            func.substr(TLS904_NUTS.nuts, 1, 2) == "IT",
            TLS904_NUTS.nuts_level == 2
        )
    )
    .order_by(TLS904_NUTS.nuts)
)

res = patstat.df(q)
res

,nuts,nuts_label
0,ITC1,Piemonte
1,ITC2,Valle d'Aosta/Vallée d'Aoste
2,ITC3,Liguria
3,ITC4,Lombardia
4,ITF1,Abruzzo
5,ITF2,Molise
6,ITF3,Campania
7,ITF4,Puglia
8,ITF5,Basilicata
9,ITF6,Calabria


## Linking TLS206_PERSON to TLS904_NUTS
To enrich the information contained in `TLS206_PERSON`, we can join it with the `TLS904_NUTS` reference table using the `NUTS` attribute. This join allows us to replace a compact regional code with meaningful contextual information, such as the NUTS level and the official name of the region.

Once the NUTS metadata have been added, the enriched dataset can be used for a variety of geographic and institutional analyses. For example, it can be used to examine how different person sectors (`PSN_SECTOR`) are distributed within a specific region (FR421, which corresponds to Bas-Rhin, in France). 

In [5]:
from epo.tipdata.patstat.database.models import TLS206_PERSON
q = (
    db.query(
        TLS206_PERSON.psn_sector,
        TLS206_PERSON.nuts,
        TLS904_NUTS.nuts_label,
        func.count(distinct(TLS206_PERSON.person_id)).label("person_count")
    )
    .join(
        TLS904_NUTS, 
        TLS904_NUTS.nuts == TLS206_PERSON.nuts
    )
    .filter(
        TLS206_PERSON.nuts == "FR421"
    )
    .group_by(
        TLS206_PERSON.psn_sector,
        TLS206_PERSON.nuts,
        TLS904_NUTS.nuts_label
    )
    .order_by(
        TLS206_PERSON.psn_sector
    )
    .distinct()
)

res = patstat.df(q)
res

,psn_sector,nuts,nuts_label,person_count
0,None,FR421,Bas-Rhin,10
1,COMPANY,FR421,Bas-Rhin,5
2,INDIVIDUAL,FR421,Bas-Rhin,2
3,UNKNOWN,FR421,Bas-Rhin,4
